<a href="https://colab.research.google.com/github/amiralito/RFD4_Proteina_colab/blob/main/RFD4_Proteina_design_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RFD4-Proteina binder & motif design

[RFD4-Proteina](https://github.com/RosettaCommons/RFD4-Proteina) (Baker Lab / IPD x NVIDIA GenAIR)
is a **flow-matching** model that co-designs sequence and all-atom structure in a single shared
chemical and conformational latent space spanning standard and modified amino acids, sugars, DNA
and RNA. One checkpoint covers binders, motif scaffolding, enzyme active sites, antibodies,
partial diffusion and inverse folding. Its "thinking" budget (`recycle_strategy`) is a recurrent
refinement schedule — more trunk passes buys a higher pass rate at lower throughput.

**Stack.** `rfd4-generate` (PyPI wheel, Python 3.12) in its own `uv` venv beside Colab's
interpreter · weights from `nvidia/RFD4-Proteina` cached on Drive · `gemmi` + `pandas` +
`matplotlib` for analysis · MolView for structures.

**Before you run anything.** Set `Runtime → Change runtime type → GPU`. An **A100 (40 GB)** is
the comfortable case; an **L4 (24 GB)** is fine for a cropped target plus a 60–120 aa binder; a
**T4 (16 GB) is the floor** and a slow one — Turing has no bfloat16 and no fused
cuEquivariance kernels, so cell 1 drops to fp16 and the torch backends automatically.

**Scope note.** Generation, conditioning metrics and partial diffusion run here. Reward-guided
search and every `*_refolding` metric need an AlphaFold 2 / AlphaFold 3 / OpenFold3 runtime
shipped as an `apptainer` image, which Colab cannot run — so **designs leave this notebook
unvalidated**. Step 11 writes the FASTA to refold in the AF3 / Protenix / OpenFold3 notebooks;
treat that refold-and-compare step as mandatory, not optional.

**Flow.** 1 GPU → 2 Drive & run name → 3 install → 4 weights → 5 target → 6 numbering &
hotspots → 7 design spec → 8 sampling settings → 9 generate → 10 design table → 11 FASTA
export → 12 plots → 13 viewer → 14 download.

## 1 · Check the GPU

Reads the device, its compute capability and the driver's CUDA version, then fixes three things
every later cell depends on: which wheel extra to install (`all-cuda12` vs `all-cuda13`), the
precision (`bf16-mixed` needs sm_80+), and whether the fused cuEquivariance kernels are usable.

In [ ]:
#@title Check allocated GPU & set build flags
import os, re, subprocess

def _smi(q):
    return subprocess.run(["nvidia-smi", f"--query-gpu={q}", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip()

_row = _smi("name,compute_cap,memory.total,driver_version").splitlines()
if not _row or not _row[0].strip():
    raise SystemExit("STOP: No GPU. Runtime -> Change runtime type -> GPU, then rerun.")

GPU_NAME, _cc, _vram, GPU_DRIVER = [x.strip() for x in _row[0].split(",")]
GPU_CC      = float(_cc)
GPU_SM      = int(round(GPU_CC * 10))
GPU_MEM_GB  = round(int(_vram.split()[0]) / 1024, 1)

# Driver CUDA version decides the wheel extra. nvidia-smi's banner carries it.
_banner = subprocess.run("nvidia-smi", shell=True, capture_output=True, text=True).stdout
_m = re.search(r"CUDA Version:\s*([0-9]+\.[0-9]+)", _banner)
DRIVER_CUDA = float(_m.group(1)) if _m else 12.0
CUDA_EXTRA  = "all-cuda13" if DRIVER_CUDA >= 13.0 else "all-cuda12"

# bfloat16 needs Ampere or newer; Turing (T4, sm_75) must run fp16.
PRECISION = "bf16-mixed" if GPU_SM >= 80 else "16-mixed"

# The fused cuEquivariance triangle/attention kernels are built for sm_80+.
KERNEL_OVERRIDES = [] if GPU_SM >= 80 else ["nn.tri_mult_backend=torch", "nn.attention_type=torch"]

print(f"{'GPU':<16}: {GPU_NAME}  ({GPU_MEM_GB} GB, sm_{GPU_SM})")
print(f"{'Driver':<16}: {GPU_DRIVER}  (CUDA {DRIVER_CUDA})")
print(f"{'Wheel extra':<16}: rfd4-proteina[{CUDA_EXTRA}]")
print(f"{'Precision':<16}: {PRECISION}")
print(f"{'Kernels':<16}: {'cuequivariance (fused)' if not KERNEL_OVERRIDES else 'torch (fallback)'}")

if GPU_SM >= 100:
    print("\nNOTE Blackwell-class GPU. The cuda13 wheel carries its kernels; if a run dies in a\n"
          "     fused kernel, add  nn.tri_mult_backend=torch nn.attention_type=torch  in step 8.")
if GPU_SM < 80:
    print("\nWARNING Turing (T4). No bf16, no fused kernels: expect several-fold slower sampling.\n"
          "        Keep BATCH_SIZE=1, crop the target hard, and keep the binder short.")
if GPU_MEM_GB < 20:
    print(f"WARNING {GPU_MEM_GB} GB VRAM. Start at BATCH_SIZE=1 and only raise it if step 9 fits.")

## 2 · Mount Drive, name this run & set output directory

Runs are timestamped folders under `<BASE_DIR>/runs`, so nothing overwrites anything. The model
weights live outside the run directory and are downloaded once. `RESUME_RUN` reattaches to an
earlier run — useful after a disconnect, because `skip_existing=true` means step 9 tops up a
partial run rather than restarting it.

In [ ]:
#@title Mount Drive, name this run & set output directory
import os, re, glob, datetime
MOUNT_DRIVE = True                                    #@param {type:"boolean"}
BASE_DIR    = "/content/drive/MyDrive/RFD4_Proteina"  #@param {type:"string"}
RUN_NAME    = "binder"                                #@param {type:"string"}
RESUME_RUN  = ""                                      #@param {type:"string"}
#  RESUME_RUN: "" = start a new run.  "latest" = reattach to the most recent run.
#  Or give a run folder name ("20261021-191003_binder") or a full path.

if MOUNT_DRIVE:
    from google.colab import drive
    _mp = "/content/drive"
    if os.path.ismount(_mp):
        print("Drive already mounted.")
    else:
        drive.mount(_mp)
elif BASE_DIR.startswith("/content/drive"):
    BASE_DIR = "/content/RFD4_Proteina"               # Drive off: do not write into a dead mount

# Shared, persistent model bundle (~1.7 GB, downloaded once, reused by every future run)
HF_HOME = f"{BASE_DIR}/hf_cache"
os.environ["HF_HOME"] = HF_HOME
os.makedirs(HF_HOME, exist_ok=True)

RUNS_DIR = f"{BASE_DIR}/runs"
os.makedirs(RUNS_DIR, exist_ok=True)

if RESUME_RUN.strip():                                # ---- resume an existing run ----
    r = RESUME_RUN.strip().rstrip("/")
    if r.lower() == "latest":
        cands = sorted(d for d in glob.glob(f"{RUNS_DIR}/*") if os.path.isdir(d))
        assert cands, f"No runs found under {RUNS_DIR} to resume."
        WORK = cands[-1]                              # timestamp prefix sorts chronologically
    else:
        WORK = r if os.path.isabs(r) else f"{RUNS_DIR}/{r}"
    assert os.path.isdir(WORK), f"Run folder not found: {WORK}"
    RUN_NAME = re.sub(r"^\d{8}-\d{6}_", "", os.path.basename(WORK))
    print("RESUMED run        :", WORK)
else:                                                 # ---- start a new run ----
    # sanitize: spaces and other characters break the Hydra command line in step 9
    clean = re.sub(r"[^A-Za-z0-9._-]", "_", RUN_NAME.strip()) or "binder"
    if clean != RUN_NAME:
        print(f"RUN_NAME sanitized: {RUN_NAME!r} -> {clean!r}")
    RUN_NAME = clean
    stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    WORK = f"{RUNS_DIR}/{stamp}_{RUN_NAME}"
    os.makedirs(WORK, exist_ok=True)
    print("NEW run            :", WORK)

INPUT_DIR = f"{WORK}/inputs"
os.makedirs(INPUT_DIR, exist_ok=True)

print("Weights (shared)   :", HF_HOME)
print("Run name           :", RUN_NAME)
print("Inputs             :", INPUT_DIR)

## Setup · Campaign status  *(run any time)*

Counts what exists in the current run and names the next step to run. Safe to execute at any
point, including in a fresh session after reattaching with `RESUME_RUN=latest`.

In [ ]:
#@title Campaign status — what's done, what's left
import os, glob
WORK = globals().get("WORK", "/content/run")

_env   = os.path.exists("/content/rfd4_env/bin/rfd4-generate")
_wts   = bool(glob.glob(os.path.join(os.environ.get("HF_HOME", ""), "hub", "models--nvidia--RFD4-Proteina*")))
_tgt   = sorted(glob.glob(f"{WORK}/inputs/*_target.cif"))
_spec  = os.path.exists(f"{WORK}/design.yaml")
_gen   = sorted(glob.glob(f"{WORK}/inference_samples/**/*_generated.cif*", recursive=True))
_met   = sorted(glob.glob(f"{WORK}/inference_metrics/per_example_metrics*.csv"))
_tab   = os.path.exists(f"{WORK}/{os.path.basename(WORK)}_designs.csv")
_fas   = sorted(glob.glob(f"{WORK}/*.fasta"))

print("run              :", WORK)
print(f"{'environment':<17}: {'ok' if _env else 'missing'}            -> step 3")
print(f"{'weights cached':<17}: {'ok' if _wts else 'missing'}            -> step 4")
print(f"{'target':<17}: {os.path.basename(_tgt[0]) if _tgt else 'none'}  -> step 5")
print(f"{'design spec':<17}: {'design.yaml' if _spec else 'none'}   -> step 7")
print(f"{'designs':<17}: {len(_gen)}                 -> step 9")
print(f"{'metrics csv':<17}: {len(_met)}")
print(f"{'design table':<17}: {'ok' if _tab else 'none'}          -> step 10")
print(f"{'fasta export':<17}: {len(_fas)}                 -> step 11")

if not _env:      print("\nNext: step 3 (install).")
elif not _wts:    print("\nNext: step 4 (weights).")
elif not _spec:   print("\nNext: steps 5-7 (target and design spec).")
elif not _gen:    print("\nNext: steps 8-9 (sampling settings, then generate).")
elif not _fas:    print("\nNext: steps 10-11 (collect designs, export FASTA).")
else:             print("\nAll stages present. Refold the FASTA elsewhere, then filter.")

## 3 · Install RFD4-Proteina

The wheel pins Python `>=3.12,<3.13` and ~7 GB of CUDA stack, so it goes into its own `uv` venv
rather than fighting Colab's interpreter; the analysis cells keep using Colab's own Python. The
repository is also shallow-cloned for its benchmark inputs and example specs. Re-running is a
no-op once the venv exists.

In [ ]:
#@title Install RFD4-Proteina  (~3 minutes, once per session)
import os, sys, shutil, subprocess, time

CLONE_REPO      = True   #@param {type:"boolean"}
#  CLONE_REPO: brings in data/benchmarks (monomer example input) and docs/examples specs.
FORCE_REINSTALL = False  #@param {type:"boolean"}

ENV_DIR   = "/content/rfd4_env"
REPO_DIR  = "/content/RFD4-Proteina"
RFD4      = f"{ENV_DIR}/bin/rfd4-generate"
RFD4_EVAL = f"{ENV_DIR}/bin/rfd4-evaluate"
RFD4_PY   = f"{ENV_DIR}/bin/python"

def sh(cmd, **kw):
    print("+", cmd)
    if subprocess.call(cmd, shell=True, **kw) != 0:
        raise RuntimeError(cmd)

def have(m):
    import importlib.util
    return importlib.util.find_spec(m) is not None

if FORCE_REINSTALL and os.path.isdir(ENV_DIR):
    shutil.rmtree(ENV_DIR)

t0 = time.time()

if not os.path.exists(RFD4):
    if shutil.which("uv") is None:
        sh(f"{sys.executable} -m pip -q install uv")
    _extra = globals().get("CUDA_EXTRA", "all-cuda12")
    sh(f"uv venv --python 3.12 {ENV_DIR}")
    # Target the interpreter explicitly. A VIRTUAL_ENV= shell prefix is NOT reliably honoured:
    # uv silently installs into Colab's own interpreter instead and still exits 0.
    # ~7 GB, mostly torch + CUDA libs. uv parallelises the download; plain pip takes ~3x longer.
    sh(f'uv pip install --python {RFD4_PY} --torch-backend=auto "rfd4-proteina[{_extra}]"')
else:
    print("rfd4-proteina already installed - skipping its venv build.")

if CLONE_REPO and not os.path.isdir(REPO_DIR):
    # --filter=blob:none keeps the clone small; the history carries old notebook outputs.
    sh(f"git clone -q --depth 1 --filter=blob:none "
       f"https://github.com/RosettaCommons/RFD4-Proteina.git {REPO_DIR}")

for _p in ("gemmi", "py3Dmol"):
    if not have(_p):
        sh(f"{sys.executable} -m pip -q install {_p}")

# Every call into the venv goes through this env. Colab exports
# MPLBACKEND=module://matplotlib_inline.backend_inline, which does not exist inside the venv,
# so matplotlib raises on import and takes rfproteina (via biotite) down with it.
RFD4_ENV = {k: v for k, v in os.environ.items()
            if k not in ("PYTHONPATH", "PYTHONHOME", "VIRTUAL_ENV")}
RFD4_ENV["MPLBACKEND"] = "Agg"

_probe = subprocess.run(
    [RFD4_PY, "-c",
     "import sys, rfproteina, rfd4_proteina as r;"
     "print(sys.version.split()[0], getattr(r,'__version__','?'))"],
    env=RFD4_ENV, capture_output=True, text=True)

if _probe.returncode != 0:
    _lst = subprocess.run(f"uv pip list --python {RFD4_PY}", shell=True,
                          capture_output=True, text=True).stdout
    raise RuntimeError(
        "The venv probe failed.\n"
        f"--- probe ---\n{_probe.stderr[-900:]}\n"
        f"--- uv pip list ({RFD4_PY}) ---\n{_lst[-900:]}\n"
        "Empty list = uv built the venv but installed nowhere; rerun with FORCE_REINSTALL=True.\n"
        "Package listed but the import fails = a dependency problem, not a missing install.")

_py, _ver = _probe.stdout.split()

print(f"\n{'rfd4-generate':<16}: {RFD4}")
print(f"{'python':<16}: {_py}")
print(f"{'rfd4-proteina':<16}: {_ver}")
print(f"{'repo':<16}: {REPO_DIR if CLONE_REPO else '(not cloned)'}")
print(f"{'elapsed':<16}: {time.time() - t0:.0f}s")

## 4 · Download model weights  *(once)*

Pulls the released bundle — denoiser plus autoencoder, ~1.7 GB — into `HF_HOME` so step 9 does
not stall on it. With Drive mounted this is a one-off across every future session. The repository
is public; `HF_TOKEN` is only needed if you have swapped in a gated checkpoint.

In [ ]:
#@title Download model weights  (once; cached on Drive for future runs)
import os, glob, subprocess

HF_CKPT = "nvidia/RFD4-Proteina"  #@param {type:"string"}
#  Pin a commit for reproducibility by passing hf_ckpt_revision=<sha> in step 8.

HF_HOME = globals().get("HF_HOME", "/content/hf_cache")
os.environ["HF_HOME"] = HF_HOME
os.makedirs(HF_HOME, exist_ok=True)

_slug  = "models--" + HF_CKPT.replace("/", "--")
_cached = glob.glob(f"{HF_HOME}/hub/{_slug}/snapshots/*/*")

if _cached:
    _bytes = sum(os.path.getsize(p) for p in
                 glob.glob(f"{HF_HOME}/hub/{_slug}/blobs/*") if os.path.isfile(p))
    print(f"Bundle already cached ({_bytes/1e9:.2f} GB) - skipping download.")
else:
    _tok = os.environ.get("HF_TOKEN", "")
    if not _tok:
        try:
            from google.colab import userdata
            _tok = userdata.get("HF_TOKEN") or ""
        except Exception:
            _tok = ""
    env = dict(globals().get("RFD4_ENV", os.environ))
    env["HF_HOME"] = HF_HOME
    if _tok:
        env["HF_TOKEN"] = _tok
    print("Downloading", HF_CKPT, "...")
    r = subprocess.run(
        [RFD4_PY, "-c",
         f"from huggingface_hub import snapshot_download as s; print(s('{HF_CKPT}'))"],
        env=env, capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip()[-2000:])
    if r.returncode != 0:
        raise RuntimeError(
            "Weight download failed. Options:\n"
            "  1. Rerun this cell - partial HF downloads resume.\n"
            "  2. If the repo is gated, add HF_TOKEN as a Colab secret (key icon, left bar).\n"
            "  3. Skip this cell: step 9 fetches the bundle itself, just more slowly.")

print("\nHF_HOME:", HF_HOME)
subprocess.run(f'du -sh "{HF_HOME}" 2>/dev/null || true', shell=True)

## 5 · Provide the target structure  *(skip for the monomer example)*

RFD4-Proteina reads `.cif`, `.cif.gz` or `.bcif` — **not** `.pdb`, so anything uploaded is
converted here with `gemmi`. The cell also drops alternate conformations, hydrogens, waters and
ligands, keeps model 1 and keeps only the chains you name.

Cropping matters: cost scales with total token count, so trimming a large target down to the
domain you are designing against is the difference between a job that fits on an L4 and one that
does not.

In [ ]:
#@title Load target (upload or fetch)
import os, subprocess, gemmi

SOURCE       = "fetch_pdb"  #@param ["fetch_pdb", "upload", "path", "none"]
PDB_ID       = "9FP6"       #@param {type:"string"}
TARGET_PATH  = ""           #@param {type:"string"}
KEEP_CHAINS  = "A"          #@param {type:"string"}
#  KEEP_CHAINS: comma-separated, e.g. "A" or "A,B". Blank keeps every chain.
STRIP_HETATM = True         #@param {type:"boolean"}

WORK = globals().get("WORK", "/content/run")
INPUT_DIR = f"{WORK}/inputs"; os.makedirs(INPUT_DIR, exist_ok=True)
TARGET_CIF = None

if SOURCE == "none":
    print("No target. Use DESIGN_TASK = monomer_example / inverse_folding / custom_yaml in step 7.")
else:
    if SOURCE == "fetch_pdb":
        raw = f"{INPUT_DIR}/{PDB_ID.lower()}_raw.cif"
        if not os.path.exists(raw):
            subprocess.run(
                f'curl -sSL -f -o "{raw}" https://files.rcsb.org/download/{PDB_ID.upper()}.cif',
                shell=True, check=True)
        assert os.path.getsize(raw) > 2000, "RCSB returned an error page, not a structure."
    elif SOURCE == "upload":
        from google.colab import files
        up = files.upload()
        _name = list(up.keys())[0]
        raw = f"{INPUT_DIR}/{_name}"
        with open(raw, "wb") as f:          # write bytes directly - works on local and Drive
            f.write(up[_name])
    else:
        raw = TARGET_PATH.strip()
        assert raw and os.path.exists(raw), f"TARGET_PATH not found: {raw!r}"

    st = gemmi.read_structure(raw)
    st.setup_entities()
    st.remove_alternative_conformations()
    st.remove_hydrogens()
    if STRIP_HETATM:
        st.remove_ligands_and_waters()
    while len(st) > 1:
        del st[1]                           # first model only

    keep = [c.strip() for c in KEEP_CHAINS.split(",") if c.strip()]
    if keep:
        _present = {c.name for c in st[0]}
        _missing = [c for c in keep if c not in _present]
        assert not _missing, f"chain(s) {_missing} not in {os.path.basename(raw)} ({sorted(_present)})"
        for _c in [c.name for c in st[0] if c.name not in keep]:
            st[0].remove_chain(_c)
    st.setup_entities()
    st.assign_label_seq_id()                # contigs index label_seq_id, so fix it now

    TARGET_CIF = f"{INPUT_DIR}/{RUN_NAME}_target.cif"
    st.make_mmcif_document().write_file(TARGET_CIF)

    _tok = sum(len([r for r in c]) for c in st[0])
    print("Target ->", TARGET_CIF)
    for c in st[0]:
        rs = [r for r in c]
        print(f"  chain {c.name:<2}: {len(rs):>4} residues   "
              f"auth {rs[0].seqid.num}-{rs[-1].seqid.num}   "
              f"label {rs[0].label_seq}-{rs[-1].label_seq}")
    print(f"  total   : {_tok} target tokens"
          + ("   (large - crop harder or expect OOM on <40 GB)" if _tok > 450 else ""))

## 6 · Residue numbering & hotspots

The gotcha that costs the most time: contig tokens and `sel(...)` selections index
**`label_seq_id`** — the contiguous 1-based numbering of the mmCIF entity — not the author
numbering printed on a PDB entry. After step 5's `assign_label_seq_id()` the two usually differ
by a constant offset, but not always (insertion codes, gaps).

Give this cell the hotspot residues in author numbering and it hands back the selection string
to paste into step 7.

In [ ]:
#@title Map auth -> label numbering, build the hotspot selection
import gemmi

INSPECT_CHAIN  = "A"            #@param {type:"string"}
HOTSPOTS_AUTH  = "480,484,487"  #@param {type:"string"}
#  HOTSPOTS_AUTH: author-numbered residues on INSPECT_CHAIN. Blank = just print the map.
HOTSPOT_LEVEL  = "sidechain_tip"  #@param ["sidechain_tip", "residue"]
#  "sidechain_tip" marks two distal sidechain atoms per residue, as the shipped PD-L1 spec does
#  (TYR -> [CG,OH], MET -> [CG,SD]): this says which FACE of the residue to engage.
#  "residue" marks every atom of the residue - looser, and it lets the binder satisfy a hotspot
#  by packing against the backbone instead of the sidechain you cared about.
SHOW_FULL_MAP  = False          #@param {type:"boolean"}

TARGET_CIF = globals().get("TARGET_CIF", None)
HOTSPOT_SELECT_AUTO = ""
HOTSPOT_LABEL = {}

if TARGET_CIF is None:
    print("No target loaded (step 5) - skipping.")
else:
    st = gemmi.read_structure(TARGET_CIF); st.setup_entities()
    chain = st[0][INSPECT_CHAIN]
    assert chain is not None, f"chain {INSPECT_CHAIN} not in this target"

    def _olc(name):
        info = gemmi.find_tabulated_residue(name)
        return info.one_letter_code.upper() if info else "X"

    a2l = {r.seqid.num: r.label_seq for r in chain}
    seq = "".join(_olc(r.name) for r in chain)
    _off = sorted({l - a for a, l in a2l.items()})

    print(f"chain {INSPECT_CHAIN}: {len(a2l)} residues, "
          f"label {min(a2l.values())}-{max(a2l.values())}")
    print("label - auth offset:", _off[0] if len(_off) == 1 else f"{_off} (not constant - check gaps)")
    print(f"\nsequence ({len(seq)} aa):")
    for i in range(0, len(seq), 60):
        print(f"  {i+1:>4} {seq[i:i+60]}")

    if SHOW_FULL_MAP:
        print("\nauth -> label")
        items = [f"{a}->{l}" for a, l in a2l.items()]
        for i in range(0, len(items), 10):
            print("  " + "  ".join(items[i:i+10]))

    # Two distal sidechain atoms per residue type: one mid-sidechain, one tip. Matches the
    # shipped PD-L1 spec's choices for the types it uses (TYR [CG,OH], MET [CG,SD]).
    TIP_ATOMS = {
        "ALA": ["CB"],        "ARG": ["CD", "NH1"],  "ASN": ["CG", "ND2"],
        "ASP": ["CG", "OD1"], "CYS": ["CB", "SG"],   "GLN": ["CD", "NE2"],
        "GLU": ["CD", "OE1"], "GLY": ["CA"],         "HIS": ["CG", "NE2"],
        "ILE": ["CG1", "CD1"],"LEU": ["CG", "CD1"],  "LYS": ["CD", "NZ"],
        "MET": ["CG", "SD"],  "PHE": ["CG", "CZ"],   "PRO": ["CG"],
        "SER": ["CB", "OG"],  "THR": ["CB", "OG1"],  "TRP": ["CD2", "CZ3"],
        "TYR": ["CG", "OH"],  "VAL": ["CB", "CG1"],
    }

    wanted = [int(x) for x in HOTSPOTS_AUTH.replace(" ", "").split(",") if x]
    if wanted:
        missing = [w for w in wanted if w not in a2l]
        assert not missing, f"residues {missing} are not in chain {INSPECT_CHAIN}"
        labels = sorted(a2l[w] for w in wanted)
        HOTSPOT_LABEL[INSPECT_CHAIN] = labels
        _by_label = {r.label_seq: r for r in chain}

        print("\nhotspots")
        _sels = []
        for w in wanted:
            lab = a2l[w]
            r = _by_label[lab]
            if HOTSPOT_LEVEL == "residue":
                print(f"  {_olc(r.name)}{w} (auth) -> label {lab}  [whole residue]")
                continue
            # Keep only tip atoms actually present: a truncated sidechain would otherwise
            # produce a selection that matches nothing.
            atoms = [a for a in TIP_ATOMS.get(r.name, []) if r.find_atom(a, "*") is not None]
            if not atoms:
                _fallback = r.find_atom("CB", "*") or r.find_atom("CA", "*")
                atoms = [_fallback.name] if _fallback is not None else []
                print(f"  {_olc(r.name)}{w} (auth) -> label {lab}  "
                      f"[no tip atoms modelled; using {atoms or 'NOTHING'}]")
            else:
                print(f"  {_olc(r.name)}{w} (auth) -> label {lab}  [{','.join(atoms)}]")
            if atoms:
                _sels.append(f"sel('{INSPECT_CHAIN}/*/{lab}/[{','.join(atoms)}]')")

        if HOTSPOT_LEVEL == "residue" or not _sels:
            HOTSPOT_SELECT_AUTO = f"sel('{INSPECT_CHAIN}/*/{labels}')"
        elif len(_sels) == 1:
            HOTSPOT_SELECT_AUTO = _sels[0]
        else:
            HOTSPOT_SELECT_AUTO = "(" + " or ".join(_sels) + ")"

        print("\n  HOTSPOT_SELECT for step 7:")
        print("  " + HOTSPOT_SELECT_AUTO)
        print("\n  Hotspots mark where the binder should engage; they are a bias, not a\n"
              "  constraint. Check obedience afterwards with the hotspot_conditioning metric.")

## 7 · Configure the design run

A design is one short YAML block: an `input` structure, a `contig` saying which parts of it to
keep and where to insert variable-length designed segments, and `conditions` annotating what the
model should respect.

| contig token | meaning |
|---|---|
| `A12-180` | keep chain A, `label_seq_id` 12–180 |
| `/0` | chain break — everything after starts a new chain |
| `BINDER:60-120` | named designed segment, length sampled in 60–120 |
| `20-30` | unnamed designed segment |

Conditions used below: `C_HOT` (hotspots), `C_CRD` (hold real coordinates), `C_SEQ` (hold real
residue identities), `C_SEQ_PROB_CYS: 0.0` (forbid cysteine). Anything you do not mention is not
provided to the model. Full grammar:
<https://rfd4-proteina.org/latest/reference/yaml-json.html>

In [ ]:
#@title Build design.yaml
import os, shutil, textwrap, gemmi

DESIGN_TASK = "binder"  #@param ["binder", "motif_scaffold", "partial_diffusion", "inverse_folding", "monomer_example"]

# ---- binder --------------------------------------------------------------------------------
BINDER_MIN_LEN  = 60   #@param {type:"slider", min:20, max:200, step:5}
BINDER_MAX_LEN  = 110  #@param {type:"slider", min:20, max:300, step:5}
HOTSPOT_SELECT  = ""   #@param {type:"string"}
#  HOTSPOT_SELECT: paste the sel(...) string printed by step 6. Blank = step 6's value if it
#  ran, otherwise an unconstrained binder anywhere on the target surface.
EXCLUDE_CYS     = True #@param {type:"boolean"}
#  EXCLUDE_CYS: forbids cysteine in the designed segment. Standard for secreted binders;
#  turn it off if you want disulfides.
TARGET_CONDITIONING = "rigid_backbone_and_sequence"  #@param ["rigid_backbone_and_sequence", "absolute_coords_and_sequence"]
#  How the target is given to the model. "rigid_backbone_and_sequence" holds its backbone
#  geometry as one internal-coordinate group and its full sequence, leaving the global pose and
#  the sidechains free - this is what the shipped PD-L1 binder spec does.
#  "absolute_coords_and_sequence" additionally pins the target where it sits in the input file.
#  Never leave the target unconditioned: anything you do not mention is not provided, so an
#  unconditioned target gets redesigned alongside the binder.

# ---- motif_scaffold ------------------------------------------------------------------------
MOTIF_CONTIG    = "A1-50, LINK:10-30, /0, BINDER:60-90"  #@param {type:"string"}
MOTIF_FIX_COORDS = "sel('A')"                            #@param {type:"string"}
MOTIF_FIX_SEQ    = "sel('A')"                            #@param {type:"string"}

# ---- partial_diffusion / inverse_folding ---------------------------------------------------
REFINE_INPUT = ""  #@param {type:"string"}
#  REFINE_INPUT: path to the .cif to refine or re-sequence. Blank uses the step 5 target.

WORK       = globals().get("WORK", "/content/run")
INPUT_DIR  = f"{WORK}/inputs"
SPEC_PATH  = f"{WORK}/design.yaml"
TARGET_CIF = globals().get("TARGET_CIF", None)
REPO_DIR   = globals().get("REPO_DIR", "/content/RFD4-Proteina")

GEN_INPUTS = None            # what goes to rfd4-generate inputs=
TASK_OVERRIDES = []          # task-specific Hydra overrides consumed by step 8

def _chain_runs(cif):
    """One contig run per chain, as label_seq_id ranges."""
    st = gemmi.read_structure(cif); st.setup_entities()
    runs = []
    for ch in st[0]:
        lab = [r.label_seq for r in ch if r.label_seq is not None]
        if lab:
            runs.append(f"{ch.name}{min(lab)}-{max(lab)}")
    return runs

def _stage(path):
    """Copy an input beside the spec so the spec's relative paths resolve."""
    dst = os.path.join(INPUT_DIR, os.path.basename(path))
    if os.path.abspath(path) != os.path.abspath(dst):
        shutil.copy(path, dst)
    return os.path.basename(dst)

if DESIGN_TASK == "monomer_example":
    import glob as _g
    cand = sorted(_g.glob(f"{REPO_DIR}/data/benchmarks/monomer/*.cif*"))
    assert cand, "Set CLONE_REPO=True in step 3 to get the shipped benchmark inputs."
    GEN_INPUTS = cand[0]
    print("Using the shipped benchmark input (annotated CIF, no spec needed):")
    print(" ", GEN_INPUTS)

elif DESIGN_TASK == "binder":
    assert TARGET_CIF, "Binder design needs a target - run step 5 first."
    assert BINDER_MAX_LEN >= BINDER_MIN_LEN, "BINDER_MAX_LEN < BINDER_MIN_LEN"

    # Binder segment first, in its own run so it gets a fresh chain id; target after.
    runs = _chain_runs(TARGET_CIF)
    contig = f"BINDER:{BINDER_MIN_LEN}-{BINDER_MAX_LEN}, /0, " + ", /0, ".join(runs)

    _st = gemmi.read_structure(TARGET_CIF); _st.setup_entities()
    tchains = [c.name for c in _st[0]]

    def _join(sels):
        return sels[0] if len(sels) == 1 else "(" + " or ".join(sels) + ")"

    sel_seq = _join([f"sel('{c}')" for c in tchains])
    sel_bb  = _join([f"sel('{c}/*/*/[N,CA,C,O]')" for c in tchains])
    sel_hot = HOTSPOT_SELECT.strip() or globals().get("HOTSPOT_SELECT_AUTO", "")

    lines = [f"{RUN_NAME}:",
             f"  input: ./inputs/{os.path.basename(TARGET_CIF)}",
             f'  contig: "{contig}"',
             "  conditions:"]

    # The target must be given to the model explicitly: anything not mentioned is not provided,
    # and an unconditioned target gets its sequence redesigned along with the binder.
    if TARGET_CONDITIONING == "rigid_backbone_and_sequence":
        # One internal-coordinate group: backbone geometry is held, global pose and sidechains
        # stay free. This is the setup the shipped PD-L1 binder spec uses.
        lines += ["    C_CRDGRP:", "      1:",
                  f'        - select: "{sel_bb} and not has_nan_coord()"']
    else:
        # Absolute coordinates: the target is pinned where it sits in the input file.
        lines += ["    C_CRD:", "      true:",
                  f'        - select: "{sel_bb} and not has_nan_coord()"']
    lines += ["    C_SEQ:", "      true:", f'        - select: "{sel_seq}"']

    if sel_hot:
        lines += ["    C_HOT:", "      true:", f'        - select: "{sel_hot}"']
    if EXCLUDE_CYS:
        lines += ["    C_SEQ_PROB_CYS:", "      0.0:",
                  '        - select: "res_name == \'BINDER\'"']

    with open(SPEC_PATH, "w") as fh:
        fh.write("\n".join(lines) + "\n")
    GEN_INPUTS = SPEC_PATH
    print("task      :", DESIGN_TASK)
    print("target    :", os.path.basename(TARGET_CIF), f"chains {tchains}")
    print("contig    :", contig)
    print("target cond:", TARGET_CONDITIONING)
    print("hotspots  :", sel_hot or "(none)")
    print("exclude C :", EXCLUDE_CYS)

elif DESIGN_TASK == "motif_scaffold":
    src = REFINE_INPUT.strip() or TARGET_CIF
    assert src, "Give a motif structure in REFINE_INPUT, or load one in step 5."
    name = _stage(src)
    lines = [f"{RUN_NAME}:",
             f"  input: ./inputs/{name}",
             f'  contig: "{MOTIF_CONTIG}"',
             "  conditions:",
             "    C_CRD:", "      true:", f'        - select: "{MOTIF_FIX_COORDS}"',
             "    C_SEQ:", "      true:", f'        - select: "{MOTIF_FIX_SEQ}"']
    with open(SPEC_PATH, "w") as fh:
        fh.write("\n".join(lines) + "\n")
    GEN_INPUTS = SPEC_PATH
    print("task      :", DESIGN_TASK)
    print("contig    :", MOTIF_CONTIG)

else:   # partial_diffusion | inverse_folding
    src = REFINE_INPUT.strip() or TARGET_CIF
    assert src, "Give the structure to refine in REFINE_INPUT."
    name = _stage(src)
    with open(SPEC_PATH, "w") as fh:
        fh.write(f"{RUN_NAME}:\n  input: ./inputs/{name}\n")
    GEN_INPUTS = SPEC_PATH
    if DESIGN_TASK == "inverse_folding":
        TASK_OVERRIDES = ["generation=inverse_folding"]
    print("task      :", DESIGN_TASK)
    print("input     :", name)
    if DESIGN_TASK == "partial_diffusion":
        print("           set PARTIAL_FRACTION in step 8 - lower = closer to the input.")

print("\ninputs=", GEN_INPUTS, "\n")
if GEN_INPUTS == SPEC_PATH:
    print(open(SPEC_PATH).read())

## 7 (advanced) · Paste a raw YAML spec  *(optional — overrides the form)*

Run this cell **instead of** the form above when you need conditions the form does not expose:
`C_RAS` burial, `C_HB` hydrogen-bond roles, `C_PISTACK` / `C_PICATION`, `C_CATH` topology,
`C_CRDGRP` internal-coordinate groups, per-token sequence probabilities, several designs in one
file, or a `global_args` block. `__TARGET__` is replaced with the step 5 target's filename.

In [ ]:
#@title Raw YAML spec (advanced)
import os, yaml

RAW_YAML = r"""
# Several designs in one file. Keys are design names; global_args sets shared defaults.
# The target placeholder below is replaced with the step 5 target's filename.

hotspot_binder:
  input: ./inputs/__TARGET__
  contig: "A1-230, /0, BINDER:70-110"
  conditions:
    C_HOT:
      true:
        - select: "sel('A/*/[112, 116, 119]')"
    C_RAS:                       # ask for a buried interface core
      BURIED:
        - select: "sel('A/*/[112, 116, 119]')"
    C_SEQ_PROB_CYS:
      0.0:
        - select: "res_name == 'BINDER'"

longer_binder:
  input: ./inputs/__TARGET__
  contig: "A1-230, /0, BINDER:110-160"
  conditions:
    C_HOT:
      true:
        - select: "sel('A/*/[112, 116, 119]')"
"""

WORK       = globals().get("WORK", "/content/run")
SPEC_PATH  = f"{WORK}/design.yaml"
TARGET_CIF = globals().get("TARGET_CIF", None)

text = RAW_YAML
if TARGET_CIF:
    text = text.replace("__TARGET__", os.path.basename(TARGET_CIF))
assert "__TARGET__" not in text, "No target loaded (step 5), but the YAML still has __TARGET__."

spec = yaml.safe_load(text)
assert isinstance(spec, dict) and spec, "The YAML did not parse into a mapping of designs."
for name, body in spec.items():
    if name == "global_args":
        continue
    assert isinstance(body, dict), f"design {name!r} is not a mapping"
    assert "input" in body or "input" in spec.get("global_args", {}), \
        f"design {name!r} has no input"

with open(SPEC_PATH, "w") as fh:
    fh.write(text.strip() + "\n")

GEN_INPUTS     = SPEC_PATH
TASK_OVERRIDES = []
DESIGN_TASK    = "custom_yaml"

print("designs:", [k for k in spec if k != "global_args"])
print("written:", SPEC_PATH, "\n")
print(open(SPEC_PATH).read())

## 8 · Sampling settings

`recycle_strategy` is the thinking budget. Every schedule runs 60 integration steps; low, medium
and high spend 120, 180 and 300 trunk passes. Going from low to high raised the published
protein-binding clustered pass rate from 7.7% to 10.6% while throughput fell from ~2,400 to ~990
designs per GPU-hour — on datacentre hardware, so expect well below that here.

Shape family matters too: `front_loaded` for protein-protein and DNA tasks, `midpeak` for
small-molecule and enzyme active-site work, `flat` for inverse folding. `thinking_low/medium/high`
alias the front-loaded presets.

`METRICS_TAGS="auto"` reads the spec and enables the oracle-free metrics its conditions make
meaningful — always `design` (clash, bond length, secondary structure, sequence adherence), plus
`hotspot_conditioning`, `internal_coordinate_conditioning` and so on where they apply. This cell
also forces `delete_failing_pre_oracle=false`: those filters exist to decide what to send to a
refolding oracle, there is no oracle here, and the upstream default would **delete** the
structure files of designs that fail one.

`HOTSPOT_SAMPLING` is the diversity knob. Off, every replicate attacks the same seed atoms. On,
each design re-draws 2–3 exposed hotspots from a patch around them, which is how the shipped VHH
campaign explores an epitope. It is passed through a composed config file rather than a CLI
override, because that is the only route the upstream examples use for sampling transforms.

In [ ]:
#@title Build the rfd4-generate command
NUM_REPLICATES   = 8   #@param {type:"integer"}
#  NUM_REPLICATES: designs per input example. Total designs = NUM_REPLICATES x (designs in spec).
BATCH_SIZE       = 1   #@param {type:"slider", min:1, max:16, step:1}
#  BATCH_SIZE: designs on the GPU at once. Test the longest input at 1 before raising it.
RECYCLE_STRATEGY = "thinking_medium"  #@param ["thinking_low", "thinking_medium", "thinking_high", "low/flat", "medium/flat", "high/flat", "low/midpeak", "medium/midpeak", "high/midpeak", "low/front_loaded", "medium/front_loaded", "high/front_loaded", "null"]
NSTEPS           = 60    #@param {type:"integer"}
GUIDANCE_W       = 1.0   #@param {type:"number"}
#  GUIDANCE_W: classifier-free guidance weight. 1.0 = off.
SEED             = 0     #@param {type:"integer"}
#  SEED: -1 leaves it unset, so every run draws different designs.
SKIP_EXISTING    = True  #@param {type:"boolean"}
#  SKIP_EXISTING: True = resume/top-up, keeping designs already in inference_samples.

PARTIAL_FRACTION = 0.4   #@param {type:"slider", min:0, max:1, step:0.05}
#  PARTIAL_FRACTION: partial_diffusion only. Fraction of the trajectory re-run; lower stays
#  closer to the input structure. Ignored by every other task.

TORCH_COMPILE    = False #@param {type:"boolean"}
#  TORCH_COMPILE: warm-up compiles 15 length buckets (several minutes) and only pays back on
#  large runs. Leave off for exploratory Colab jobs.
DETERMINISTIC_ODE = False #@param {type:"boolean"}
#  DETERMINISTIC_ODE: ODE sampling instead of the stochastic default. Same seed -> same design.

METRICS_TAGS = "auto"  #@param ["auto", "none", "design", "design,hotspot_conditioning", "design,hotspot_conditioning,internal_coordinate_conditioning,interchain_interactions"]
#  "auto" reads the spec and enables every oracle-free tag its conditions make meaningful.
#  The "design" tag alone is worth having: it turns on clash, bond-length, secondary-structure
#  and sequence-adherence metrics, all of which run without a refolding oracle. Every
#  *_refolding tag needs an AF2/AF3/OpenFold3 apptainer runtime, which Colab cannot provide.

PREVALIDATE_INPUTS = True  #@param {type:"boolean"}
#  Sanity-check each unique input before generating. Upstream default is false, so a bad contig
#  otherwise costs you the model load plus sampling before it fails.
SAVE_CHAIN_OF_INTEREST = True  #@param {type:"boolean"}
#  Also write a binder-only CIF per design (<id>_chain.cif). Needed for FoldSeek clustering,
#  and handy for loading just the designed chain.

HOTSPOT_SAMPLING = False  #@param {type:"boolean"}
#  Re-draw the hotspots for EVERY design from an exposed patch around the ones you gave, instead
#  of reusing your exact seed set. This is how the shipped VHH campaign gets epitope diversity:
#  without it, num_replicates designs all attack the same three atoms. Needs C_HOT in the spec.
HOTSPOTS_PER_DESIGN_MIN   = 2     #@param {type:"slider", min:1, max:8, step:1}
HOTSPOTS_PER_DESIGN_MAX   = 3     #@param {type:"slider", min:1, max:8, step:1}
HOTSPOT_PATCH_DIAMETER_A  = 15.0  #@param {type:"number"}
#  Patch size around the seed centroid. Larger = more epitope drift per design.
HOTSPOT_MIN_DISTANCE_A    = 5.0   #@param {type:"number"}
HOTSPOT_MIN_SASA          = 1.0   #@param {type:"number"}
#  Minimum atom SASA, so sampled hotspots stay solvent-exposed and therefore reachable.
HOTSPOT_PATCH_SHAPE       = "sphere"  #@param ["sphere", "oval"]
#  "oval" is a surface-aligned ellipsoid - better for a flat or elongated epitope.

EXTRA_OVERRIDES = ""  #@param {type:"string"}
#  EXTRA_OVERRIDES: space-separated Hydra key=value tokens, appended last so they win.

assert "GEN_INPUTS" in globals() and GEN_INPUTS, "Run step 7 (or the advanced YAML cell) first."

WORK = globals().get("WORK", "/content/run")

OVERRIDES = [
    f"inputs={GEN_INPUTS}",
    f"out_dir={WORK}",
    f"num_replicates={NUM_REPLICATES}",
    f"batch_size={BATCH_SIZE}",
    f"skip_existing={SKIP_EXISTING}",
    f"generation.args.nsteps={NSTEPS}",
    f"generation.args.guidance_w={GUIDANCE_W}",
    f"precision={globals().get('PRECISION', 'bf16-mixed')}",
    f"recycle_strategy={RECYCLE_STRATEGY}",
]
OVERRIDES += globals().get("TASK_OVERRIDES", [])
OVERRIDES += globals().get("KERNEL_OVERRIDES", [])

if SEED >= 0:
    OVERRIDES.append(f"seed={SEED}")
if not TORCH_COMPILE:
    OVERRIDES.append("compile_lengths=null")
if DETERMINISTIC_ODE:
    OVERRIDES += ["generation.model.bb_ca.gt.mode=ode",
                  "generation.model.local_latents.gt.mode=ode"]
if globals().get("DESIGN_TASK") == "partial_diffusion" and PARTIAL_FRACTION > 0:
    OVERRIDES.append(f"generation.args.partial_trajectory_fraction_steps={PARTIAL_FRACTION}")
OVERRIDES.append(f"prevalidate_inputs={PREVALIDATE_INPUTS}")
OVERRIDES.append(f"save_chain_of_interest={SAVE_CHAIN_OF_INTEREST}")

# Pre-oracle filters exist to decide what is worth sending to a refolding oracle. There is no
# oracle here, and delete_failing_pre_oracle defaults to TRUE upstream - which would delete the
# structure files of designs that fail a filter. Keep every design and filter in step 10 instead.
OVERRIDES.append("delete_failing_pre_oracle=false")

# ---- which conditions does the spec actually set? ----
SPEC_CONDITIONS, SPEC_MULTICHAIN = set(), False
if str(GEN_INPUTS).endswith((".yaml", ".yml", ".json")):
    import yaml as _yaml
    _spec = _yaml.safe_load(open(GEN_INPUTS)) or {}
    _full2short = {"coordinate": "C_CRD", "sequence": "C_SEQ", "hotspot": "C_HOT",
                   "rasa-binned": "C_RAS", "CoordinateGroup": "C_CRDGRP", "index": "C_IDX"}
    for _name, _body in _spec.items():
        if _name == "global_args" or not isinstance(_body, dict):
            continue
        for _k in (_body.get("conditions") or {}):
            SPEC_CONDITIONS.add(_full2short.get(str(_k), str(_k)))
        if "/0" in str(_body.get("contig", "")):
            SPEC_MULTICHAIN = True

if METRICS_TAGS == "auto":
    _tags = ["design"]                       # oracle-free, always meaningful
    if "C_HOT" in SPEC_CONDITIONS:
        _tags.append("hotspot_conditioning")
    if "C_CRDGRP" in SPEC_CONDITIONS:
        _tags.append("internal_coordinate_conditioning")
    if "C_CRD" in SPEC_CONDITIONS:
        _tags.append("3d_coordinate_conditioning")
    if "C_RAS" in SPEC_CONDITIONS:
        _tags.append("rasa_conditioning")
    if SPEC_MULTICHAIN:
        _tags.append("interchain_interactions")
    if globals().get("DESIGN_TASK") == "partial_diffusion":
        _tags.append("partial_diffusion")
    METRICS_TAGS_RESOLVED = ",".join(dict.fromkeys(_tags))
else:
    METRICS_TAGS_RESOLVED = METRICS_TAGS

if METRICS_TAGS_RESOLVED == "none":
    OVERRIDES.append("~metrics")
else:
    OVERRIDES.append(f"metrics_tags=[{METRICS_TAGS_RESOLVED}]")

# ---- hotspot patch sampling needs a config file, not a CLI override ----
EXPERIMENT_CONFIG = None
if HOTSPOT_SAMPLING and globals().get("DESIGN_TASK") in ("binder", "custom_yaml"):
    assert "C_HOT" in SPEC_CONDITIONS, (
        "HOTSPOT_SAMPLING replaces SEED hotspots with a nearby patch, so the spec must already\n"
        "set C_HOT. Give hotspots in step 6/7, or turn sampling off.")
    EXPERIMENT_CONFIG = f"{WORK}/rfd4_colab.yaml"
    _lo, _hi = sorted((int(HOTSPOTS_PER_DESIGN_MIN), int(HOTSPOTS_PER_DESIGN_MAX)))
    with open(EXPERIMENT_CONFIG, "w") as fh:
        fh.write(
            "# Written by the notebook. Composes the installed inference config, then adds the\n"
            "# per-replicate hotspot patch sampler (as examples/antibody_design/vhh.yaml does).\n"
            "defaults:\n  - inference\n  - _self_\n\n"
            "hydra:\n  searchpath:\n    - file://${oc.env:PROJECT_ROOT}/configs\n\n"
            "generation:\n  dataset:\n    sampling_transforms:\n"
            "      - _target_: rfproteina.transforms.hotspot_sampling.SampleNearbyHotspots\n"
            f"        num_hotspots: [{_lo}, {_hi}]\n"
            f"        min_distance: {HOTSPOT_MIN_DISTANCE_A}\n"
            f"        patch_diameter: {HOTSPOT_PATCH_DIAMETER_A}\n"
            f"        min_atom_sasa: {HOTSPOT_MIN_SASA}\n"
            f"        shape: {HOTSPOT_PATCH_SHAPE}\n")

if EXTRA_OVERRIDES.strip():
    OVERRIDES += EXTRA_OVERRIDES.split()

print("metrics tags :", METRICS_TAGS_RESOLVED or "(none)",
      f"   [{'auto from spec: ' + ', '.join(sorted(SPEC_CONDITIONS)) if METRICS_TAGS == 'auto' else 'manual'}]")
if EXPERIMENT_CONFIG:
    print("hotspot patch: sampled per design ->", EXPERIMENT_CONFIG)
    print()
    print(open(EXPERIMENT_CONFIG).read())
print("\nrfd4-generate \\")
if EXPERIMENT_CONFIG:
    print(f"    --config-path {WORK} --config-name rfd4_colab \\")
print("    " + " \\\n    ".join(OVERRIDES))

## 9 · Generate

Streams the log to the console and to `<run>/generate.log` at once. Designs land in
`<run>/inference_samples/<example_id>/<example_id>_generated.cif`, metrics in
`<run>/inference_metrics/per_example_metrics*.csv`.

Because `skip_existing=true`, re-running after a disconnect tops the run up instead of starting
over — reattach first with `RESUME_RUN=latest` in step 2.

In [ ]:
#@title Run RFD4-Proteina
import os, glob, subprocess, time

VERBOSE = False   # True = show every INFO line from Hydra/Lightning

assert "OVERRIDES" in globals(), "Run step 8 first - it builds OVERRIDES."
WORK = globals().get("WORK", "/content/run")
RFD4 = globals().get("RFD4", "/content/rfd4_env/bin/rfd4-generate")
assert os.path.exists(RFD4), "rfd4-generate not found - run step 3."

LOG_PATH = f"{WORK}/generate.log"
# RFD4_ENV (from step 3) drops PYTHONPATH/PYTHONHOME/VIRTUAL_ENV and forces MPLBACKEND=Agg,
# without which Colab's inline matplotlib backend breaks every import inside the venv.
env = dict(globals().get("RFD4_ENV", os.environ))
env["MPLBACKEND"] = "Agg"
env["HF_HOME"] = globals().get("HF_HOME", env.get("HF_HOME", "/content/hf_cache"))
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # reduce fragmentation OOMs

_cfg = globals().get("EXPERIMENT_CONFIG")
cmd = [RFD4]
if _cfg:
    # Hotspot sampling lives in a composed config, so point Hydra at it. CLI overrides still win.
    cmd += ["--config-path", os.path.dirname(_cfg),
            "--config-name", os.path.splitext(os.path.basename(_cfg))[0]]
cmd += OVERRIDES
print("+ " + " ".join(cmd) + "\n", flush=True)

t0 = time.time()
_buf = []
with open(LOG_PATH, "w") as log:
    p = subprocess.Popen(cmd, cwd=WORK, env=env, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        _buf.append(line); log.write(line)
        low = line.lower()
        if (VERBOSE
                or any(k in low for k in ("error", "traceback", "failed", "exception",
                                          "out of memory", "warning", 'file "', "^^^"))
                or any(k in line for k in ("Generat", "Sampling", "Saved", "Wrote", "Metric",
                                           "example", "Seed", "Completed"))
                or "%|" in line or "it/s" in line or "s/it" in line
                or "INFO" not in line):
            print(line, end="")
    p.wait()

if any("unambiguously determine chain of interest" in l for l in _buf):
    print("\nNOTE The chain-of-interest heuristic was ambiguous, so metrics and any *_coi column\n"
          "     scored the first chain in sorted order - for a binder run that is usually the\n"
          "     TARGET, not the binder. It picks the least sequence-conditioned protein chain,\n"
          "     so this means the target carries no C_SEQ: rebuild the spec with step 7, which\n"
          "     conditions the target's sequence and geometry.")

_gen = sorted(glob.glob(f"{WORK}/inference_samples/**/*_generated.cif*", recursive=True))
print(f"\nexit {p.returncode} | {time.time()-t0:.0f}s | {len(_gen)} design(s) on disk")
print("log:", LOG_PATH)

if p.returncode != 0:
    _tail = "".join(_buf[-40:])
    print("\n--- last lines ---\n" + _tail)
    print("""
Common causes
  CUDA out of memory             BATCH_SIZE=1, crop the target harder, shorten the binder range
  cuequivariance kernel error    EXTRA_OVERRIDES: nn.tri_mult_backend=torch nn.attention_type=torch
  bf16 unsupported               precision=16-mixed (step 1 sets this on T4 automatically)
  'contig needs a preserved...'  every contig needs at least one chain/range token
  residue not found in select    selections index label_seq_id, not auth numbering - see step 6
  all designs fail a filter      the pre-oracle tag gate is looser than the metric it reads""")

## 10 · Collect designs & rank

Parses every generated CIF with `gemmi` and builds one row per design: the designed chain's
length and sequence, radius of gyration, hydrophobic fraction, cysteine count, and the number of
designed residues contacting the target. Any `per_example_metrics*.csv` from step 8's tags is
merged on the design id.

None of these are validation. They are cheap filters for deciding what is worth refolding.

In [ ]:
#@title Build the design table
import os, glob, math
import numpy as np, pandas as pd, gemmi
from IPython.display import display

CONTACT_CUTOFF_A = 5.0  #@param {type:"number"}
#  CONTACT_CUTOFF_A: a designed residue counts as interface if its CB (CA for Gly) is within
#  this distance of any target heavy atom.

pd.set_option("display.max_colwidth", 80)
WORK = globals().get("WORK", "/content/run")
TARGET_CIF = globals().get("TARGET_CIF", None)

cifs = sorted(glob.glob(f"{WORK}/inference_samples/**/*_generated.cif*", recursive=True))
assert cifs, f"No designs under {WORK}/inference_samples. Did step 9 finish?"

TARGET_CHAINS = set()
if TARGET_CIF and os.path.exists(TARGET_CIF):
    _t = gemmi.read_structure(TARGET_CIF); _t.setup_entities()
    TARGET_CHAINS = {c.name for c in _t[0]}

def _olc(name):
    info = gemmi.find_tabulated_residue(name)
    return info.one_letter_code.upper() if info else "X"

def _rep_atom(res):
    return res.find_atom("CB", "*") or res.find_atom("CA", "*")

rows, STRUCTURES = [], {}
for f in cifs:
    try:
        st = gemmi.read_structure(f)
        st.setup_entities(); st.remove_hydrogens()
        # Drop the crystal cell: otherwise NeighborSearch counts symmetry images as contacts.
        st.cell = gemmi.UnitCell(); st.spacegroup_hm = "P 1"
        st.setup_cell_images()
    except Exception as e:
        print("skip", os.path.basename(f), "-", e); continue

    design_id = os.path.basename(f).split("_generated")[0]
    STRUCTURES[design_id] = f

    chains = {c.name: [r for r in c] for c in st[0]}
    des_names = [n for n in chains if n not in TARGET_CHAINS] or list(chains)
    des_res = [r for n in des_names for r in chains[n]]

    _ca = [r.find_atom("CA", "*") for r in des_res]
    ca = np.array([a.pos.tolist() for a in _ca if a is not None])
    rg = float(np.sqrt(((ca - ca.mean(0)) ** 2).sum(1).mean())) if len(ca) else math.nan
    seq = "".join(_olc(r.name) for r in des_res)

    n_iface = 0
    if TARGET_CHAINS & set(chains):
        ns = gemmi.NeighborSearch(st, CONTACT_CUTOFF_A).populate()
        for r in des_res:
            at = _rep_atom(r)
            if at is None:
                continue
            marks = ns.find_atoms(at.pos, "\0", radius=CONTACT_CUTOFF_A)
            if any(st[0][m.chain_idx].name in TARGET_CHAINS for m in marks):
                n_iface += 1

    rows.append({
        "design_id": design_id,
        "designed_chains": ",".join(des_names),
        "design_len": len(des_res),
        "interface_res": n_iface,
        "frac_interface": round(n_iface / max(len(des_res), 1), 3),
        "radius_gyration_A": round(rg, 2) if rg == rg else "NA",
        "frac_hydrophobic": round(sum(c in "AVLIMFWYC" for c in seq) / max(len(seq), 1), 3),
        "n_cys": seq.count("C"),
        "sequence": seq,
        "path": f,
    })

df = pd.DataFrame(rows)
df = df.sort_values(["interface_res", "design_len"], ascending=[False, True]).reset_index(drop=True)

_mets = sorted(glob.glob(f"{WORK}/inference_metrics/per_example_metrics*.csv"))
if _mets:
    mdf = pd.concat([pd.read_csv(m) for m in _mets], ignore_index=True)
    key = next((c for c in mdf.columns if c.lower() in
                ("example_id", "design_id", "name", "id", "sample_id")), None)
    if key:
        keep = [c for c in mdf.columns if not c.startswith("generation.")
                and c not in ("ckpt_path", "ckpt_name", "seed")]
        df = df.merge(mdf[keep].rename(columns={key: "design_id"}), on="design_id", how="left")
        print(f"Merged {len(mdf)} metric row(s) from {len(_mets)} file(s).")
    else:
        print("Metrics CSV found but no recognisable id column - not merged.")

TABLE_PATH = f"{WORK}/{RUN_NAME}_designs.csv"
df.to_csv(TABLE_PATH, index=False)

#  Put the metrics that actually gate a design first: clashes and sequence adherence (from the
#  "design" tag) and hotspot obedience beat any of the geometry computed above.
_preferred = ["design_id", "design_len", "interface_res", "frac_interface"]
_preferred += [c for c in df.columns if "clash" in c.lower()]
_preferred += [c for c in df.columns
               if any(k in c.lower() for k in ("sequence_adherence", "hotspot_obedience",
                                               "motif_allatom_rmsd"))]
_preferred += ["radius_gyration_A", "frac_hydrophobic", "n_cys"]
front = [c for c in dict.fromkeys(_preferred) if c in df.columns]
rest  = [c for c in df.columns if c not in front + ["sequence", "path"]]

print(f"\n{len(df)} design(s). Saved {TABLE_PATH}\n")
display(df[front + rest].head(40))

if len(df):
    print(f"\ndesign length  : {df['design_len'].min()}-{df['design_len'].max()} aa "
          f"(median {int(df['design_len'].median())})")
    if TARGET_CHAINS:
        print(f"interface res  : {df['interface_res'].min()}-{df['interface_res'].max()} "
              f"(median {int(df['interface_res'].median())})")
        print("                 ~15-30 contacting residues is a typical designed PPI; very low\n"
              "                 means the binder drifted off the target, very high often means\n"
              "                 it wrapped around a crop edge rather than the real epitope.")
    _hard = [c for c in df.columns if "clash" in c.lower() or "adherence" in c.lower()]
    if _hard:
        print("\nHard checks present:", ", ".join(_hard[:4]))
        print("  A nonzero sequence_adherence.n_violations means the model did not reproduce the\n"
              "  sequence you conditioned - on a binder run that is the target being redesigned.")
    print("\nStill no validation: nothing here says a design folds or binds. Refold first.")

## 11 · Export sequences for refolding

RFD4-Proteina hands back a sequence and a structure together, so the real filter is
refold-and-compare: predict the designed sequence (with the target, for a binder), then check
interface confidence and the RMSD back to the design model.

This writes a FASTA ready for the AF3 / Protenix / OpenFold3 notebooks — one record per designed
chain, with the target chains repeated alongside each design so every entry is a complete complex
job.

In [ ]:
#@title Write FASTA for AF3 / Protenix / OpenFold3
import os, gemmi

INCLUDE_TARGET = True  #@param {type:"boolean"}
#  INCLUDE_TARGET: repeat the target chain(s) under each design so each entry is a complex job.
MIN_LEN        = 30    #@param {type:"integer"}
TOP_N          = 0     #@param {type:"integer"}
#  TOP_N: export only the top N rows of the step 10 table (already sorted). 0 = all.

assert "df" in globals() and len(df), "Run step 10 first."
WORK = globals().get("WORK", "/content/run")
TARGET_CIF = globals().get("TARGET_CIF", None)

def _olc(name):
    info = gemmi.find_tabulated_residue(name)
    return info.one_letter_code.upper() if info else "X"

target_seqs = {}
if INCLUDE_TARGET and TARGET_CIF and os.path.exists(TARGET_CIF):
    _t = gemmi.read_structure(TARGET_CIF); _t.setup_entities()
    for c in _t[0]:
        target_seqs[c.name] = "".join(_olc(r.name) for r in c)

sub = df if TOP_N <= 0 else df.head(TOP_N)
FASTA_PATH = f"{WORK}/{RUN_NAME}_designs.fasta"
n = 0
with open(FASTA_PATH, "w") as fh:
    for _, r in sub.iterrows():
        if r["design_len"] < MIN_LEN:
            continue
        fh.write(f">{r['design_id']}|design|{r['design_len']}aa\n{r['sequence']}\n")
        for cname, cseq in target_seqs.items():
            fh.write(f">{r['design_id']}|target_{cname}|{len(cseq)}aa\n{cseq}\n")
        n += 1

print(f"{n} design(s) -> {FASTA_PATH}")
if target_seqs:
    print("target chains included:", ", ".join(f"{k} ({len(v)} aa)" for k, v in target_seqs.items()))
print("\nNext: refold each design+target in the AF3 / Protenix / OpenFold3 notebook, then keep\n"
      "designs with a confident interface AND a low RMSD back to the design model here.\n"
      "Interface confidence alone is not enough - a refold can be confident about a different pose.")
print("\n--- head ---")
print("".join(open(FASTA_PATH).readlines()[:6]))

## 12 · Summary plots

Distributions across the run, plus a Cb distance map for one design with chain boundaries marked
— the quickest way to see whether the designed chain actually packs against the target or merely
sits beside it.

In [ ]:
#@title Distributions + contact map
import os
import numpy as np, matplotlib.pyplot as plt, gemmi

CONTACT_MAP_DESIGN = "first"  #@param {type:"string"}
#  CONTACT_MAP_DESIGN: "first" (top of the step 10 table) or an exact design_id.
MAP_CUTOFF_A       = 8.0      #@param {type:"number"}
SAVE_FIGURES       = True     #@param {type:"boolean"}

assert "df" in globals() and len(df), "Run step 10 first."
WORK = globals().get("WORK", "/content/run")
FIG_DIR = f"{WORK}/figures"; os.makedirs(FIG_DIR, exist_ok=True)

_base = [c for c in ("design_len", "interface_res", "radius_gyration_A", "frac_hydrophobic")
         if c in df.columns]
_extra = [c for c in df.columns
          if c not in _base + ["design_id", "designed_chains", "sequence", "path",
                               "n_cys", "frac_interface"]
          and np.issubdtype(df[c].dtype, np.number) and df[c].notna().any()]
panels = _base + _extra[:4]

ncol = min(4, max(1, len(panels)))
nrow = int(np.ceil(len(panels) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3.4 * ncol, 2.8 * nrow), squeeze=False)
for ax, col in zip(axes.ravel(), panels):
    v = pd.to_numeric(df[col], errors="coerce").dropna().values
    if not len(v):
        ax.axis("off"); continue
    ax.hist(v, bins=min(20, max(5, len(v) // 2)), color="#2b6cb0", edgecolor="white")
    ax.axvline(np.median(v), color="#c0392b", ls="--", lw=1.2)
    ax.set_xlabel(col.replace("_", " ")); ax.set_ylabel("designs")
for ax in axes.ravel()[len(panels):]:
    ax.axis("off")
fig.suptitle(f"{RUN_NAME} - {len(df)} designs (dashed = median)", y=1.0, fontsize=11)
plt.tight_layout()
if SAVE_FIGURES:
    plt.savefig(f"{FIG_DIR}/{RUN_NAME}_distributions.png", dpi=130, bbox_inches="tight")
plt.show()

pick = df.iloc[0]["design_id"] if CONTACT_MAP_DESIGN == "first" else CONTACT_MAP_DESIGN
if pick not in STRUCTURES:
    print(f"{pick} is not in this run - skipping the contact map.")
else:
    st = gemmi.read_structure(STRUCTURES[pick]); st.setup_entities(); st.remove_hydrogens()
    res, bounds, mids, names, off = [], [], [], [], 0
    for ch in st[0]:
        rs = [r for r in ch]
        res += rs; mids.append(off + len(rs) / 2); names.append(ch.name)
        off += len(rs); bounds.append(off)
    coords = np.array([(r.find_atom("CB", "*") or r.find_atom("CA", "*")).pos.tolist()
                       for r in res if (r.find_atom("CB", "*") or r.find_atom("CA", "*"))])
    d = np.linalg.norm(coords[:, None] - coords[None], axis=-1)

    plt.figure(figsize=(5.6, 4.8))
    im = plt.imshow(d, cmap="viridis_r", vmin=0, vmax=25)
    plt.contour(d < MAP_CUTOFF_A, levels=[0.5], colors="white", linewidths=0.4)
    for b in bounds[:-1]:
        plt.axhline(b - 0.5, color="k", lw=0.8); plt.axvline(b - 0.5, color="k", lw=0.8)
    plt.xticks(mids, names); plt.yticks(mids, names)
    plt.colorbar(im, label="Cb-Cb distance (Angstrom)")
    plt.title(f"{pick} - distance map, chains marked")
    plt.xlabel("residue"); plt.ylabel("residue")
    plt.tight_layout()
    if SAVE_FIGURES:
        plt.savefig(f"{FIG_DIR}/{RUN_NAME}_{pick}_distmap.png", dpi=130, bbox_inches="tight")
    plt.show()
    print("Off-diagonal blocks are the interface. A designed chain with an empty off-diagonal\n"
          "block is not touching the target, whatever its other numbers say.")

## 13 · Model viewer (MolView)

Target in grey, designed chain coloured, hotspots as red sticks. Pick a design from the dropdown.
For anything serious, open the CIF in ChimeraX — the command is printed below the viewer.

In [ ]:
#@title MolView — browse designs
import os, sys, subprocess

try:
    import molview as mv
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "molview"], check=False)
    import molview as mv

import ipywidgets as widgets
from IPython.display import display, clear_output
try:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
except Exception:
    pass

assert "STRUCTURES" in globals() and STRUCTURES, "Run step 10 first."

_ids = list(df["design_id"]) if "df" in globals() else sorted(STRUCTURES)
design_dd = widgets.Dropdown(options=_ids, description="Design:",
                             layout=widgets.Layout(width="70%"))
color_dd  = widgets.Dropdown(options=["chain", "rainbow", "secondary"], value="chain",
                             description="Color:", layout=widgets.Layout(width="40%"))
out = widgets.Output()

def _render(*_):
    with out:
        clear_output(wait=True)
        did = design_dd.value
        cif = STRUCTURES[did]
        row = df.set_index("design_id").loc[did] if "df" in globals() else None
        if row is not None:
            print(f"{did} | {row['design_len']} aa designed | "
                  f"{row['interface_res']} interface residues | "
                  f"{row['frac_hydrophobic']} hydrophobic")
        try:
            v = mv.view(width=760, height=540, panel=True)
            with open(cif) as f:
                v.addModel(f.read())
            v.setColorMode(color_dd.value)
            v.show()
        except Exception as e:
            print("MolView render failed:", e)
            print("Open this file in ChimeraX instead:", cif)
        print("\nChimeraX:  open", cif)
        _hot = globals().get("HOTSPOT_LABEL", {})
        if _hot:
            for ch, ids in _hot.items():
                print(f"           select /{ch}:{','.join(str(i) for i in ids)}  "
                      f"then  show sel atoms  (hotspots, label numbering)")

design_dd.observe(_render, names="value")
color_dd.observe(_render, names="value")
display(widgets.VBox([design_dd, color_dd, out]))
_render()

## 14 · Download results

Designs already live on Drive when step 2 mounted it, so there is nothing to download in that
case. Without Drive, this zips the run and hands it over.

In [ ]:
#@title Zip & download outputs
import os, shutil

WORK = globals().get("WORK", "/content/run")

if WORK.startswith("/content/drive/"):
    print("Outputs already persisted on Drive at:", WORK, "\n(no download needed)")
    for root, _dirs, files_ in os.walk(WORK):
        if root.count(os.sep) - WORK.count(os.sep) < 2 and files_:
            print(" ", os.path.relpath(root, WORK), f"({len(files_)} files)")
else:
    _zip = f"/content/{RUN_NAME}_rfd4"
    shutil.make_archive(_zip, "zip", WORK)
    zp = _zip + ".zip"
    print("Zipped:", zp, f"({os.path.getsize(zp)/1e6:.1f} MB)")
    try:
        from google.colab import files
        files.download(zp)
    except Exception as e:
        print("Download manually from the Files panel:", zp, "|", e)

---

### Notes & next steps

- **Outputs.** `<run>/inference_samples/<example_id>/<example_id>_generated.cif` is the design;
  `<example_id>_input.cif` is the conditioning sidecar the model actually saw — open it when a
  contig did not do what you expected. Metrics land in `<run>/inference_metrics/`, this
  notebook's own products (`*_designs.csv`, `*_designs.fasta`, `figures/`, `generate.log`) sit
  at the run root.

- **Nothing here is validated.** Colab cannot run the AF2/AF3/OpenFold3 apptainer images, so
  every `*_refolding` metric tag and every oracle-scored search checkpoint is off. A design that
  looks good in step 10 may not fold. Refold the step 11 FASTA elsewhere and filter on interface
  confidence *and* RMSD back to the design model before ordering anything.

- **What the oracle-free metrics do buy you.** The `design` tag reports clashes, bond-length
  geometry, secondary-structure distribution and sequence adherence; `hotspot_conditioning`
  reports hotspot obedience; `internal_coordinate_conditioning` reports motif all-atom RMSD, which
  with a `C_CRDGRP` target tells you the rigid group was held. Treat these as necessary, not
  sufficient: they reject bad designs rather than confirming good ones.

- **Pre-oracle filters are disabled on purpose.** They exist to decide what is worth sending to a
  refolding oracle, and upstream `delete_failing_pre_oracle=true` would delete the structure files
  of designs that fail one. With no oracle in the loop, step 8 forces it to `false` so you keep
  every design and filter in step 10 yourself.

- **Reward-guided search** also needs those oracles for its usual configurations. Structure-only
  scoring (for instance the `sm_binder_hbond_search` preset, which ranks on hydrogen-bond and
  hydrophobic contact terms rather than a refold) does run without them — pass
  `inference_experiment=sm_binder_hbond_search` through `EXTRA_OVERRIDES` in step 8 if that
  fits your objective. Anything referencing `oracle.*` will fail here.

- **Numbering.** Contigs and `sel(...)` index `label_seq_id`, not author numbering. Step 6
  converts, and prints the offset so you can tell when it is not constant.

- **Out of memory.** In order: `BATCH_SIZE=1`, crop the target to the domain you care about,
  shorten the binder range, then drop to `thinking_low`. `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`
  is already set in step 9.

- **Reproducibility.** Fix `SEED` and set `DETERMINISTIC_ODE=True` for repeatable designs, and
  pin the checkpoint with `hf_ckpt_revision=<commit>` through `EXTRA_OVERRIDES`. The default
  stochastic sampler will not reproduce even at a fixed seed.

- **Scale.** Published throughput is roughly 1,000–2,400 designs per GPU-hour on datacentre
  hardware; a Colab L4 sits well below that and a T4 further still. Use this notebook to get a
  spec right on a handful of designs, then move the campaign to a SLURM array — the repository
  ships `scripts/launch_generate.sh` and the `gen_njobs` / `job_id` options for exactly that.

- **Docs:** [quickstart](https://rfd4-proteina.org/latest/quickstart.html) ·
  [input preparation](https://rfd4-proteina.org/latest/how-to/input_prep/index.html) ·
  [YAML/JSON reference](https://rfd4-proteina.org/latest/reference/yaml-json.html) ·
  [CLI options](https://rfd4-proteina.org/latest/reference/cli-options.html) ·
  [search](https://rfd4-proteina.org/latest/how-to/search.html)

- Cite: Corley, Didi, Kazman, Brent *et al.* (2026). *Biomolecular design through latent
  reasoning with RFD4-Proteina*. Preprint. <https://rfd4-proteina.org/assets/paper/RFD4-Proteina.pdf>
  — code BSD-3-Clause, weights under NVIDIA's terms on Hugging Face.